# 第15章：长上下文与高效推理

## 本章目标
- 理解 4K → 128K → 1M 上下文扩展背后的工程原理
- 掌握 RoPE 缩放、Ring Attention、Speculative Decoding、PagedAttention

## 前置知识
- 复习第11章：RoPE 位置编码
- 复习第9章：推理优化基础（KV Cache）

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch matplotlib
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## 1. RoPE 缩放策略

RoPE 默认只支持训练时的最大长度 $L_{train}$。要在 $L_{new} > L_{train}$ 上使用，需要缩放策略：

### 三种主流方法

| 方法 | 核心思想 | 优势 |
|------|---------|------|
| Position Interpolation (PI) | 线性压缩位置到训练范围 | 简单，无微调也能用 |
| NTK-aware interpolation | 调整频率基数 base | 保留高频信息 |
| YaRN | NTK + 注意力温度调整 | 综合效果最好 |

参考：[YaRN](https://arxiv.org/abs/2309.00071), [NTK-aware](https://arxiv.org/abs/2306.15595)

In [ ]:
import torch
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

# 基础 RoPE 函数（来自 Ch11）
def precompute_rope_freqs(head_dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, freqs)
    freqs = torch.cat([freqs, freqs], dim=-1)
    return freqs.cos(), freqs.sin()

def apply_rotary_emb(x, cos, sin):
    d = x.shape[-1]
    x1 = x[..., :d // 2]
    x2 = x[..., d // 2:]
    rotated = torch.cat([-x2, x1], dim=-1)
    return x * cos + rotated * sin

def simple_attention(q, k, v, head_dim):
    """无位置编码的基础注意力"""
    att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
    att = F.softmax(att, dim=-1)
    return att @ v

print("RoPE 基础函数已加载")

In [ ]:
# 策略1: Position Interpolation (PI)
def rope_pi(head_dim, seq_len, scale_factor, base=10000.0):
    """Position Interpolation：线性压缩位置。
    
    将 [0, L_new] 压缩到 [0, L_train]，
    保持旋转角度在训练范围内。
    """
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    # 关键：位置除以 scale_factor
    t = torch.arange(seq_len).float() / scale_factor
    freqs = torch.outer(t, freqs)
    freqs = torch.cat([freqs, freqs], dim=-1)
    return freqs.cos(), freqs.sin()

# 策略2: NTK-aware interpolation
def rope_ntk(head_dim, seq_len, scale_factor, base=10000.0):
    """NTK-aware：动态调整 base 频率。
    
    不压缩位置，而是增大 base 使得高频分量被保留。
    base_new = base * scale_factor ^ (dim / (dim - 2))
    """
    base_new = base * (scale_factor ** (head_dim / (head_dim - 2)))
    freqs = 1.0 / (base_new ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len).float()
    freqs = torch.outer(t, freqs)
    freqs = torch.cat([freqs, freqs], dim=-1)
    return freqs.cos(), freqs.sin()

# 策略3: YaRN
def rope_yarn(head_dim, seq_len, scale_factor, base=10000.0, temp_factor=1.0):
    """YaRN：NTK + 注意力温度调整。
    
    在 NTK-aware 基础上，对注意力分数施加温度缩放。
    temp_factor > 1 使得注意力更集中在近处。
    """
    base_new = base * (scale_factor ** (head_dim / (head_dim - 2)))
    freqs = 1.0 / (base_new ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len).float()
    freqs = torch.outer(t, freqs)
    freqs = torch.cat([freqs, freqs], dim=-1)
    return freqs.cos(), freqs.sin()

print("三种缩放策略函数已定义")

In [ ]:
# 可视化对比三种缩放策略
head_dim, train_len, test_len = 64, 64, 256
scale_factor = test_len / train_len  # 4x

q = torch.randn(1, 1, test_len, head_dim)
k = torch.randn(1, 1, test_len, head_dim)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 基准：原始 RoPE（超出训练长度）
cos_base, sin_base = precompute_rope_freqs(head_dim, test_len)
cos_b = cos_base[:test_len].unsqueeze(0).unsqueeze(0)
sin_b = sin_base[:test_len].unsqueeze(0).unsqueeze(0)
q_r = apply_rotary_emb(q, cos_b, sin_b)
k_r = apply_rotary_emb(k, cos_b, sin_b)
att_base = F.softmax((q_r @ k_r.transpose(-2, -1)) / math.sqrt(head_dim), dim=-1)
axes[0,0].imshow(att_base[0,0].detach().numpy(), cmap='Blues', aspect='auto')
axes[0,0].set_title('原始 RoPE（超出训练长度）')

# PI
cos_pi, sin_pi = rope_pi(head_dim, test_len, scale_factor)
cos_p = cos_pi[:test_len].unsqueeze(0).unsqueeze(0)
sin_p = sin_pi[:test_len].unsqueeze(0).unsqueeze(0)
q_r = apply_rotary_emb(q, cos_p, sin_p)
k_r = apply_rotary_emb(k, cos_p, sin_p)
att_pi = F.softmax((q_r @ k_r.transpose(-2, -1)) / math.sqrt(head_dim), dim=-1)
axes[0,1].imshow(att_pi[0,0].detach().numpy(), cmap='Blues', aspect='auto')
axes[0,1].set_title('Position Interpolation')

# NTK-aware
cos_ntk, sin_ntk = rope_ntk(head_dim, test_len, scale_factor)
cos_n = cos_ntk[:test_len].unsqueeze(0).unsqueeze(0)
sin_n = sin_ntk[:test_len].unsqueeze(0).unsqueeze(0)
q_r = apply_rotary_emb(q, cos_n, sin_n)
k_r = apply_rotary_emb(k, cos_n, sin_n)
att_ntk = F.softmax((q_r @ k_r.transpose(-2, -1)) / math.sqrt(head_dim), dim=-1)
axes[1,0].imshow(att_ntk[0,0].detach().numpy(), cmap='Blues', aspect='auto')
axes[1,0].set_title('NTK-aware Interpolation')

# YaRN
cos_yarn, sin_yarn = rope_yarn(head_dim, test_len, scale_factor, temp_factor=1.0)
cos_y = cos_yarn[:test_len].unsqueeze(0).unsqueeze(0)
sin_y = sin_yarn[:test_len].unsqueeze(0).unsqueeze(0)
q_r = apply_rotary_emb(q, cos_y, sin_y)
k_r = apply_rotary_emb(k, cos_y, sin_y)
att_yarn = F.softmax((q_r @ k_r.transpose(-2, -1)) / math.sqrt(head_dim), dim=-1)
axes[1,1].imshow(att_yarn[0,0].detach().numpy(), cmap='Blues', aspect='auto')
axes[1,1].set_title('YaRN')

for ax in axes.flat:
    ax.set_xlabel('Key 位置')
    ax.set_ylabel('Query 位置')
plt.suptitle(f'RoPE 缩放策略对比（训练长度={train_len}, 测试长度={test_len}）')
plt.tight_layout()
plt.show()

## 2. 训练时上下文扩展

实际模型通常采用渐进式策略：

| 阶段 | 上下文长度 | 训练 tokens |
|------|-----------|-------------|
| Phase 1 | 4K | 大部分 (T) |
| Phase 2 | 32K | 部分 (0.1T) |
| Phase 3 | 128K | 少量 (10B) |

**为什么有效：**
- 模型先学会短文本的模式（语法、语义）
- 再扩展到长文本（文档级连贯性、长程依赖）
- 大部分计算在短上下文阶段完成，长上下文阶段只做微调

参考：Llama 3.1 技术报告，[DeepSeek-V3](https://arxiv.org/abs/2412.19437) 训练细节

## 3. Ring Attention

### 核心思想

当序列太长（如 1M tokens），单个 GPU 放不下完整的 Q/K/V 矩阵。

Ring Attention 的解决方案：
1. 把序列切成 N 段，分配到 N 个 GPU
2. 每个 GPU 计算自己那段 Q 与当前 K/V 的注意力
3. K/V 沿着「环形」传递给下一个 GPU
4. 累积注意力结果

参考：[Ring Attention](https://arxiv.org/abs/2310.01889) (Liu et al., 2023)

In [ ]:
# Ring Attention 概念演示
def ring_attention_demo(q_chunks, k_chunks, v_chunks, n_devices=4):
    """模拟 Ring Attention 的计算过程。
    
    实际实现需要 NVLink/NVSwitch 进行设备间通信，
    这里用 CPU 模拟逻辑。
    """
    n_chunks = len(q_chunks)
    head_dim = q_chunks[0].shape[-1]
    outputs = [torch.zeros_like(q_chunks[i]) for i in range(n_chunks)]
    
    for step in range(n_devices):
        print(f"Step {step}: ", end="")
        for device_id in range(n_devices):
            # 当前设备持有的 KV 块（环形传递）
            kv_id = (device_id + step) % n_devices
            q = q_chunks[device_id]  # (1, 1, chunk_len, head_dim)
            k = k_chunks[kv_id]
            v = v_chunks[kv_id]
            
            # 注意：这里简化了 causal mask
            att = F.softmax((q @ k.transpose(-2, -1)) / math.sqrt(head_dim), dim=-1)
            outputs[device_id] = outputs[device_id] + att @ v
            
            print(f"Dev{device_id}←KV{kv_id}", end="  ")
        print()
    
    return outputs

# 模拟 4 个设备，每设备处理 16 个 token
chunk_len, head_dim = 16, 32
q_chunks = [torch.randn(1, 1, chunk_len, head_dim) for _ in range(4)]
k_chunks = [torch.randn(1, 1, chunk_len, head_dim) for _ in range(4)]
v_chunks = [torch.randn(1, 1, chunk_len, head_dim) for _ in range(4)]

print("=== Ring Attention 模拟 ===\n")
outputs = ring_attention_demo(q_chunks, k_chunks, v_chunks, n_devices=4)
print(f"\n每个设备输出 shape: {outputs[0].shape}")
print("实际实现中，KV 块通过 NVLink 在设备间环形传递，计算和通信重叠")

## 4. Speculative Decoding（投机解码）

### 核心思想

大模型推理慢（每 token 需要一次完整前向传播）。

Speculative Decoding 用一个小模型做「草稿」，大模型做「验证」：
1. Draft 模型快速生成 $k$ 个 token
2. Target 模型一次前向传播评估所有 $k$ 个 token
3. 接受与 target 分布一致的 token，拒绝不一致的

**关键性质：** 最终输出与直接用大模型完全一致（无损加速）。

参考：[Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2302.01318) (Leviathan et al., 2023)

In [ ]:
def speculative_decode(draft_model_call, target_model_call, prompt_ids, k=4, max_new_tokens=20):
    """概念演示 Speculative Decoding。
    
    draft_model_call/target_model_call: 接受 token ids，返回 logits 的函数
    """
    generated = list(prompt_ids)
    total_draft_tokens = 0
    accepted_tokens = 0
    
    while len(generated) - len(prompt_ids) < max_new_tokens:
        # Step 1: Draft 模型生成 k 个 token
        draft_tokens = []
        draft_probs = []
        current = list(generated)
        for _ in range(k):
            logits = draft_model_call(current)
            probs = F.softmax(logits[-1], dim=-1)
            token = torch.multinomial(probs, 1).item()
            draft_tokens.append(token)
            draft_probs.append(probs)
            current.append(token)
        total_draft_tokens += k
        
        # Step 2: Target 模型一次前向传播评估所有 k 个 token
        target_logits = target_model_call(current)
        
        # Step 3: 逐个验证
        n_accepted = 0
        for i in range(k):
            target_probs = F.softmax(target_logits[len(generated) + i - len(prompt_ids) + len(prompt_ids) - 1], dim=-1)
            # 简化验证：如果 draft 选的是 target 概率最高的 token，就接受
            target_top1 = target_probs.argmax().item()
            if draft_tokens[i] == target_top1:
                n_accepted += 1
            else:
                # 拒绝：用 target 的选择替代，丢弃后续 draft tokens
                break
        
        accepted_tokens += n_accepted
        # 接受的 token + target 的修正 token
        correction_token = F.softmax(target_logits[len(generated) + n_accepted - len(prompt_ids) + len(prompt_ids) - 1], dim=-1).argmax().item()
        for t in draft_tokens[:n_accepted]:
            generated.append(t)
        if n_accepted < k:
            generated.append(correction_token)
            
        if len(generated) >= len(prompt_ids) + max_new_tokens:
            break
    
    acceptance_rate = accepted_tokens / total_draft_tokens if total_draft_tokens > 0 else 0
    return generated, acceptance_rate

print("Speculative Decoding 函数已定义（概念演示版本）")

In [ ]:
# 用随机 logits 模拟 speculative decoding
vocab_size = 100

def draft_call(ids):
    """模拟小模型：随机 logits"""
    return torch.randn(len(ids), vocab_size)

def target_call(ids):
    """模拟大模型：稍好的 logits（有一定偏好）"""
    return torch.randn(len(ids), vocab_size)

# 运行演示
prompt = [1, 5, 10, 15]
result, acc_rate = speculative_decode(draft_call, target_call, prompt, k=4, max_new_tokens=16)

print(f"Prompt: {prompt}")
print(f"生成结果: {result}")
print(f"生成长度: {len(result) - len(prompt)} tokens")
print(f"Draft 接受率: {acc_rate:.1%}")
print(f"\n理论加速比 = 1 / (1 - acceptance_rate * (k-1)/k)，k={4}")
print(f"实际加速取决于 draft 模型和 target 模型的相似度")

## 5. vLLM PagedAttention

### KV Cache 内存管理问题

推理时每个请求都需要 KV Cache，但：
- 不同请求长度不同 → 内存碎片严重
- 预分配最大长度 → 大量浪费

### PagedAttention：借鉴操作系统虚拟内存

| 操作系统概念 | PagedAttention 对应 |
|-------------|-------------------|
| 虚拟内存页 | 虚拟 KV block |
| 物理内存页 | 物理 KV block |
| 页表 | Block Table |
| 按需分配 | 动态分配 block |

**核心优势：**
1. 非连续内存分配 → 消除碎片
2. 按需分配 → 不浪费预留空间
3. 共享 block → Prefix caching（相同前缀的请求共享 KV cache）

参考：[vLLM](https://arxiv.org/abs/2309.06180) (Kwon et al., 2023), [github.com/vllm-project/vllm](https://github.com/vllm-project/vllm)

In [ ]:
class BlockManager:
    """简化的 PagedAttention Block Manager。
    
    管理物理 block 池，为每个序列动态分配 block。
    """
    def __init__(self, total_blocks, block_size=16):
        self.block_size = block_size
        self.total_blocks = total_blocks
        # 可用 block 集合
        self.free_blocks = set(range(total_blocks))
        # 每个序列的 block 表：seq_id → [物理 block id 列表]
        self.block_tables = {}
    
    def allocate(self, seq_id, num_tokens):
        """为序列分配足够的 block。"""
        num_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        if seq_id not in self.block_tables:
            self.block_tables[seq_id] = []
        
        current_blocks = len(self.block_tables[seq_id])
        for _ in range(num_blocks_needed - current_blocks):
            if not self.free_blocks:
                raise RuntimeError("OOM: 没有可用的物理 block")
            block_id = self.free_blocks.pop()
            self.block_tables[seq_id].append(block_id)
    
    def free(self, seq_id):
        """释放序列占用的所有 block。"""
        if seq_id in self.block_tables:
            for block_id in self.block_tables[seq_id]:
                self.free_blocks.add(block_id)
            del self.block_tables[seq_id]
    
    def get_block_table(self, seq_id):
        """获取序列的 block 表（类似页表）。"""
        return self.block_tables.get(seq_id, [])
    
    def status(self):
        """当前内存使用状态。"""
        used = self.total_blocks - len(self.free_blocks)
        return f"Block 使用: {used}/{self.total_blocks} ({used/self.total_blocks:.1%})"

# 演示
bm = BlockManager(total_blocks=100, block_size=16)
print(f"初始状态: {bm.status()}")

# 模拟 3 个不同长度的请求
bm.allocate("req_1", num_tokens=50)   # 需要 4 个 block
bm.allocate("req_2", num_tokens=128)  # 需要 8 个 block
bm.allocate("req_3", num_tokens=30)   # 需要 2 个 block
print(f"3 个请求后: {bm.status()}")

print(f"\nBlock Tables:")
for seq_id, blocks in bm.block_tables.items():
    print(f"  {seq_id}: 物理块 {blocks}")

# 释放一个请求
bm.free("req_2")
print(f"\n释放 req_2 后: {bm.status()}")
print("释放的 block 可以立即给新请求使用，没有内存碎片！")

In [ ]:
# 对比：连续分配 vs PagedAttention
print("=== 内存使用对比 ===\n")

# 假设：3 个请求，最大长度 512 tokens，block_size=16
max_seq_len = 512
block_size = 16
seq_lengths = [50, 128, 30]

# 方法1：预分配最大长度（传统方式）
traditional_blocks = sum((max_seq_len + block_size - 1) // block_size for _ in seq_lengths)
traditional_tokens = traditional_blocks * block_size

# 方法2：PagedAttention 按需分配
paged_blocks = sum((l + block_size - 1) // block_size for l in seq_lengths)
paged_tokens = paged_blocks * block_size

print(f"{'方法':<20s} {'Blocks':>10s} {'Tokens':>10s} {'利用率':>10s}")
print(f"{'预分配最大长度':<20s} {traditional_blocks:>10d} {traditional_tokens:>10d} {sum(seq_lengths)/traditional_tokens:>10.1%}")
print(f"{'PagedAttention':<20s} {paged_blocks:>10d} {paged_tokens:>10d} {sum(seq_lengths)/paged_tokens:>10.1%}")
print(f"\n节省: {(1 - paged_blocks/traditional_blocks)*100:.0f}% blocks")

## 练习

1. 修改 `scale_factor` 为 8 和 16，观察不同缩放策略的注意力分布变化
2. 在 Ring Attention 演示中添加 causal mask，思考跨设备边界如何处理
3. 用真实小模型（如 Ch1 的 baby GPT）替换随机 logits，运行 Speculative Decoding
4. 修改 `BlockManager` 支持 prefix caching（相同前缀的请求共享 block）

## 延伸阅读

- [YaRN](https://arxiv.org/abs/2309.00071) — 高效 RoPE 上下文扩展
- [NTK-aware Interpolation](https://arxiv.org/abs/2306.15595) — NTK 感知插值
- [Ring Attention](https://arxiv.org/abs/2310.01889) — 环形注意力 (Liu et al., 2023)
- [Speculative Decoding](https://arxiv.org/abs/2302.01318) — 投机解码 (Leviathan et al., 2023)
- [vLLM / PagedAttention](https://arxiv.org/abs/2309.06180) — 分页注意力 (Kwon et al., 2023)
- [vLLM 源码](https://github.com/vllm-project/vllm) — 工业级实现参考